# Unit economics of a revolving credit line

Synthetic portfolio, illustrative parameters (see `unit_economics/params.py`). The questions:

1. How do losses season by cohort, and when is a vintage's loss mostly visible?
2. What is the expected loss per line in each risk segment?
3. Does the price cover funding, expected loss and opex in every segment?
4. What does one line earn over its life, and when does it pay back its acquisition cost?
5. How sensitive is per-line contribution to the loss rate, the interest rate and the limit?

In [1]:
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd

from unit_economics import Params
from unit_economics import economics as E
from unit_economics import portfolio as P

pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
p = Params()
CHARTS = "charts"
INK, GRID = "#1f2933", "#d9dee3"
SEG_COLORS = {"A": "#2f6f8f", "B": "#d08c2e", "C": "#b5473a"}
plt.rcParams.update({"axes.edgecolor": GRID, "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK,
                     "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": GRID, "grid.linewidth": 0.6, "font.size": 10})

## 1. Cohort loss curves

Twelve monthly vintages; younger ones have less history. Cumulative loss is net of recoveries, as a share of the vintage's originated balance.

In [2]:
book = P.originate(p)
curves = P.cohort_loss_curves(book, p.lgd)
fig, ax = plt.subplots(figsize=(7.5, 4))
for c in curves.columns:
    ax.plot(curves.index, curves[c] * 100, color="#2f6f8f", alpha=0.25 + 0.65 * (1 - c / curves.columns.max()), lw=1.4)
ax.text(0.99, 0.04, "darker = older vintage", transform=ax.transAxes, ha="right", color=INK, fontsize=9)
ax.set(xlabel="Months on book", ylabel="Cumulative net loss, % of originated balance",
       title="Losses season over the first year; most of a vintage's loss shows by month 9")
fig.tight_layout(); fig.savefig(f"{CHARTS}/loss_curves.png", dpi=160)
curves.loc[[3, 6, 9, 12, 18, 24]].mul(100).round(2).iloc[:, :4]

,0,1,2,3
3,1.100,1.110,0.940,1.230
6,2.820,2.990,2.110,2.580
9,3.640,4.510,3.800,3.560
12,4.030,5.360,4.610,3.820
18,4.550,5.360,4.610,4.200
24,4.550,NaN,NaN,NaN


## 2. Expected loss by segment

EL = PD × LGD × EAD, annual, with EAD as the average drawn balance.

In [3]:
el = E.expected_loss_by_segment(p)
el

,pd_12m,lgd,ead,el_per_line,el_rate
segment,,,,,
A,0.020,0.850,"1,800.000",30.600,0.017
B,0.060,0.850,"1,375.000",70.125,0.051
C,0.140,0.850,840.000,99.960,0.119


## 3. Pricing against the cost stack

Annual rates on drawn balance. Opex spreads acquisition over the horizon and adds servicing.

In [4]:
stack = E.pricing_vs_loss(p)
fig, ax = plt.subplots(figsize=(7.5, 4))
ax.set_axisbelow(True); ax.grid(axis="x", visible=False)
bottom = 0
parts = [("funding", "#9aa5b1"), ("expected_loss", "#b5473a"), ("opex", "#d08c2e"), ("margin", "#2f6f8f")]
for col, color in parts:
    ax.bar(stack.index, stack[col] * 100, bottom=bottom, color=color, label=col.replace("_", " "), width=0.55)
    bottom = bottom + stack[col] * 100
ax.axhline(p.apr * 100, color=INK, lw=1, ls="--")
ax.text(2.33, p.apr * 100 + 0.6, f"APR {p.apr:.0%}", ha="right", color=INK)
ax.set(ylabel="% of drawn balance, annual", xlabel="Segment",
       title="The same price leaves 19 points of margin in A and under 6 in C")
ax.legend(frameon=False, ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.15))
fig.tight_layout(); fig.savefig(f"{CHARTS}/pricing_vs_loss.png", dpi=160)
stack

,apr,funding,expected_loss,opex,margin
segment,,,,,
A,0.360,0.120,0.017,0.030,0.193
B,0.360,0.120,0.051,0.039,0.150
C,0.360,0.120,0.119,0.064,0.057


## 4. One line over its life

Expected monthly cash flows for an average line per segment; the acquisition cost is paid in month 1.

In [5]:
fig, ax = plt.subplots(figsize=(7.5, 4))
for s in p.segments:
    cf = E.line_cashflows(p, s)
    ax.plot(cf.index, cf["cumulative"], color=SEG_COLORS[s.name], lw=2, label=f"Segment {s.name}")
    m = E.payback_month(cf)
    if m:
        ax.scatter([m], [cf.loc[m, "cumulative"]], color=SEG_COLORS[s.name], zorder=3)
ax.axhline(0, color=INK, lw=0.8)
ax.set(xlabel="Month", ylabel="Cumulative contribution per line", title="Payback: month 2 in A, month 11 in C")
ax.legend(frameon=False)
fig.tight_layout(); fig.savefig(f"{CHARTS}/payback.png", dpi=160)
E.line_summary(p)

,revenue,funding,losses,opex,contribution,payback_month
segment,,,,,,
A,"1,276.340",-425.450,-32.110,-107.270,711.510,2
B,945.700,-315.230,-72.180,-105.850,452.430,3
C,543.750,-181.250,-99.010,-103.150,160.340,11


## 5. Sensitivity

Contribution per line over the 24-month horizon, by annual net loss rate, APR and limit (utilisation 55%).

In [6]:
sens = E.sensitivity(p)
sens.index = pd.MultiIndex.from_tuples([(f"{l:.0%}", f"{r:.0%}") for l, r in sens.index],
                                       names=["annual loss rate", "APR"])
sens.columns = [f"limit {c:,.0f}" for c in sens.columns]
sens.style.format("{:,.0f}").background_gradient(cmap="RdBu", vmin=-1000, vmax=1000)

## Reading

- A vintage shows most of its loss by month 9; judging a cohort at month 3 understates it by more than half.
- Segment C carries seven times the expected loss rate of A at the same price; its margin is thin enough that small changes in loss or opex erase it.
- Small limits are the weak spot: at a 24% APR a 1,000 line loses money from a 5% loss rate, because the acquisition cost is fixed and the balance is small.
- Rate moves contribution more than loss does across this grid; limit size moves it most.